# Hiệu quả cải thiện sau từng bước tối ưu

Notebook này trực quan hóa bảng ablation của hệ thống RAG SGU trên 570 câu hỏi và 1.840 passages. Mục tiêu là tạo các biểu đồ tăng trưởng hiệu năng qua từng phiên bản V1 -> V6.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
})

OUT_DIR = Path("outputs") / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR

## 1. Dữ liệu ablation

In [ ]:
metrics = ["Recall@1", "Recall@3", "Recall@5", "MRR@10"]

data = [
    {
        "Version": "V1",
        "Step": "Lexical baseline",
        "Retrieval Mode": "BM25 Okapi thô",
        "Recall@1": 0.2842,
        "Recall@3": 0.4158,
        "Recall@5": 0.4825,
        "MRR@10": 0.3428,
    },
    {
        "Version": "V2.1",
        "Step": "Dense PhoBERT",
        "Retrieval Mode": "PhoBERT Bi-Encoder thô",
        "Recall@1": 0.0825,
        "Recall@3": 0.1246,
        "Recall@5": 0.1584,
        "MRR@10": 0.0963,
    },
    {
        "Version": "V2.2",
        "Step": "Dense Legal HF",
        "Retrieval Mode": "Vietnam-Legal-HF thô",
        "Recall@1": 0.3263,
        "Recall@3": 0.4851,
        "Recall@5": 0.5526,
        "MRR@10": 0.3947,
    },
    {
        "Version": "V2.3",
        "Step": "Dense BGE-M3",
        "Retrieval Mode": "BGE-M3 Multilingual thô",
        "Recall@1": 0.4158,
        "Recall@3": 0.6025,
        "Recall@5": 0.6807,
        "MRR@10": 0.4891,
    },
    {
        "Version": "V3",
        "Step": "Off-shelf CE rerank",
        "Retrieval Mode": "BGE-M3 + Cross-Encoder rerank",
        "Recall@1": 0.4912,
        "Recall@3": 0.6842,
        "Recall@5": 0.7246,
        "MRR@10": 0.5842,
    },
    {
        "Version": "V4",
        "Step": "FT Bi-Encoder",
        "Retrieval Mode": "Fine-tuned Bi-Encoder bi_bge_m3_ft",
        "Recall@1": 0.5246,
        "Recall@3": 0.7298,
        "Recall@5": 0.7825,
        "MRR@10": 0.6218,
    },
    {
        "Version": "V5",
        "Step": "FT Bi + FT CE",
        "Retrieval Mode": "FT Bi-Encoder + FT Cross-Encoder",
        "Recall@1": 0.5912,
        "Recall@3": 0.7860,
        "Recall@5": 0.8211,
        "MRR@10": 0.6825,
    },
    {
        "Version": "V6",
        "Step": "Hybrid RAG + RRF + FT CE",
        "Retrieval Mode": "BM25 + FT Bi-Encoder + RRF + FT CE rerank",
        "Recall@1": 0.6368,
        "Recall@3": 0.8158,
        "Recall@5": 0.8544,
        "MRR@10": 0.7282,
    },
]

df = pd.DataFrame(data)
df["Version Step"] = df["Version"] + "\n" + df["Step"]

display(
    df[["Version", "Step", "Retrieval Mode", *metrics]].style.format({m: "{:.2%}" for m in metrics})
)

## 2. Tính mức cải thiện

In [ ]:
baseline = df.loc[0, metrics]
final = df.loc[df["Version"].eq("V6"), metrics].iloc[0]

analysis_df = df.copy()
for metric in metrics:
    analysis_df[f"{metric} vs V1 (pp)"] = (analysis_df[metric] - baseline[metric]) * 100
    analysis_df[f"{metric} step lift (pp)"] = analysis_df[metric].diff() * 100

summary = pd.DataFrame({
    "Metric": metrics,
    "V1 baseline": [baseline[m] for m in metrics],
    "V6 final": [final[m] for m in metrics],
    "Absolute lift (pp)": [(final[m] - baseline[m]) * 100 for m in metrics],
    "Relative lift": [(final[m] / baseline[m] - 1) for m in metrics],
})

display(
    summary.style.format({
        "V1 baseline": "{:.2%}",
        "V6 final": "{:.2%}",
        "Absolute lift (pp)": "{:+.2f}",
        "Relative lift": "{:+.1%}",
    })
)

## 3. Biểu đồ tăng trưởng hiệu năng tổng thể

In [ ]:
colors = {
    "Recall@1": "#2563eb",
    "Recall@3": "#0891b2",
    "Recall@5": "#16a34a",
    "MRR@10": "#f97316",
}

fig, ax = plt.subplots(figsize=(12, 6.2))
x = np.arange(len(df))

for metric in metrics:
    ax.plot(
        x,
        df[metric],
        marker="o",
        linewidth=2.4,
        markersize=6,
        color=colors[metric],
        label=metric,
    )
    for i, value in enumerate(df[metric]):
        if i in [0, len(df) - 1]:
            ax.annotate(
                f"{value:.1%}",
                xy=(i, value),
                xytext=(0, 8),
                textcoords="offset points",
                ha="center",
                fontsize=8,
                color=colors[metric],
            )

ax.axvspan(len(df) - 1.35, len(df) - 0.65, color="#dcfce7", alpha=0.75, label="Phiên bản tốt nhất")
ax.set_title("Biểu đồ tăng trưởng hiệu năng qua từng bước tối ưu")
ax.set_ylabel("Giá trị metric")
ax.set_xticks(x)
ax.set_xticklabels(df["Version Step"], rotation=0, ha="center")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, 0.92)
ax.legend(ncol=5, loc="upper left", frameon=True)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)

fig.tight_layout()
fig_path = OUT_DIR / "performance_growth_line.png"
fig.savefig(fig_path, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## 4. Mức tăng sau từng bước tối ưu

In [ ]:
step_lift = analysis_df.loc[1:, ["Version", "Step", "Recall@5 step lift (pp)", "MRR@10 step lift (pp)"]]

fig, ax = plt.subplots(figsize=(12, 5.6))
bar_width = 0.36
x = np.arange(len(step_lift))

r5 = step_lift["Recall@5 step lift (pp)"]
mrr = step_lift["MRR@10 step lift (pp)"]

bars1 = ax.bar(x - bar_width / 2, r5, width=bar_width, color="#16a34a", label="Recall@5")
bars2 = ax.bar(x + bar_width / 2, mrr, width=bar_width, color="#f97316", label="MRR@10")

ax.axhline(0, color="#374151", linewidth=1)
ax.set_title("Đóng góp tăng/giảm hiệu năng sau mỗi bước")
ax.set_ylabel("Thay đổi so với bước trước, điểm phần trăm")
ax.set_xticks(x)
ax.set_xticklabels(step_lift["Version"] + "\n" + step_lift["Step"], rotation=0, ha="center")
ax.legend(frameon=True)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)

for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        va = "bottom" if height >= 0 else "top"
        offset = 0.8 if height >= 0 else -0.8
        ax.annotate(
            f"{height:+.1f}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, offset),
            textcoords="offset points",
            ha="center",
            va=va,
            fontsize=8,
        )

fig.tight_layout()
fig_path = OUT_DIR / "stepwise_lift_recall5_mrr10.png"
fig.savefig(fig_path, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## 5. So sánh baseline V1 và bản tốt nhất V6

In [ ]:
compare = df[df["Version"].isin(["V1", "V6"])].set_index("Version")[metrics].T

fig, ax = plt.subplots(figsize=(9.5, 5.4))
x = np.arange(len(metrics))
bar_width = 0.34

baseline_bars = ax.bar(x - bar_width / 2, compare["V1"], width=bar_width, color="#64748b", label="V1 baseline")
final_bars = ax.bar(x + bar_width / 2, compare["V6"], width=bar_width, color="#0f766e", label="V6 optimized")

ax.set_title("Từ baseline đến hệ thống tối ưu V6")
ax.set_ylabel("Giá trị metric")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_ylim(0, 0.94)
ax.legend(frameon=True)
ax.grid(axis="y", alpha=0.25)
ax.grid(axis="x", visible=False)

for idx, metric in enumerate(metrics):
    lift_pp = (compare.loc[metric, "V6"] - compare.loc[metric, "V1"]) * 100
    ax.annotate(
        f"+{lift_pp:.1f} pp",
        xy=(idx, compare.loc[metric, "V6"]),
        xytext=(0, 8),
        textcoords="offset points",
        ha="center",
        fontsize=9,
        fontweight="bold",
        color="#0f766e",
    )

fig.tight_layout()
fig_path = OUT_DIR / "baseline_vs_v6.png"
fig.savefig(fig_path, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## 6. Xuất bảng số liệu cho báo cáo

In [ ]:
csv_path = OUT_DIR.parent / "ablation_performance_growth_summary.csv"
export_df = analysis_df.copy()
pp_cols = [col for col in export_df.columns if col.endswith("(pp)")]
export_df[metrics] = export_df[metrics].round(4)
export_df[pp_cols] = export_df[pp_cols].round(2)
export_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Kết luận nhanh:")
for _, row in summary.iterrows():
    print(
        f"- {row['Metric']}: {row['V1 baseline']:.2%} -> {row['V6 final']:.2%} "
        f"({row['Absolute lift (pp)']:+.2f} pp, {row['Relative lift']:+.1%})"
    )

print(f"\nSaved CSV: {csv_path}")